# Notebook 06 — Model Comparison and Architecture Selection

**AI Interview Assistant · Machine Learning Pipeline, Stage 6 of 9**

---

## Purpose

Choose one architecture to carry forward, and make the choice **defensible**.

## The selection problem

Ranking by validation loss alone is the obvious approach and the wrong one. It
ignores three things that matter for a system that has to run in production:

1. **Generalisation** — a model with the lowest loss but a widening
   train/validation gap has memorised, not learned.
2. **Efficiency** — this model serves live interview traffic. A 2 % loss
   improvement bought with 4× the parameters is a bad trade.
3. **Stability** — a model whose validation curve oscillates has not converged,
   and its best epoch may be noise rather than skill.

So selection is **multi-criteria**, with weights declared before the scores are
seen.

## Criteria and weights

| Criterion | Weight | Direction | What it measures |
|---|---|---|---|
| Validation loss | 0.45 | lower better | predictive quality on unseen data |
| Generalisation gap | 0.25 | lower better | learned vs memorised |
| Parameter efficiency | 0.15 | lower better | inference cost |
| Convergence stability | 0.15 | lower better | is the best epoch trustworthy? |

Validation loss dominates, but cannot win alone. Every criterion is
**min–max normalised** to [0, 1] so the weights mean what they say.

## Constraint: validation data only

The test split is not read here. Selecting on test data and then reporting the
test score is circular — the score would measure the selection, not the model.

## Outputs

- `reports/model_selection.json` — the winner and the full scorecard
- `reports/figures/06_*.png`

---

In [ ]:
NOTEBOOK_ID = 6

# ─────────────────────────────────────────────────────────────────────────────
# Step 0 — Environment bootstrap
#
# Locates the project workspace so this notebook runs unchanged in Google Colab,
# a local Jupyter server, or VS Code. Every later step resolves its paths from
# WORKSPACE_DIR, so nothing below depends on where the notebook was opened.
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

def locate_workspace() -> Path:
    """Return the ml-service directory, whatever environment we are in."""
    # 1. Google Colab: mount Drive so checkpoints survive a runtime restart.
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        ws = Path("/content/drive/MyDrive/ai-interview-system/ml-service")
        ws.mkdir(parents=True, exist_ok=True)
        print("Environment      : Google Colab (Drive mounted)")
        return ws
    except ImportError:
        pass

    # 2. Local: walk up from the notebook until we find the ml-service root,
    #    identified by the dataset directory it must contain.
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "dataset").is_dir() and (candidate / "notebooks").is_dir():
            print("Environment      : local")
            return candidate
    print("Environment      : local (fallback to cwd)")
    return here

WORKSPACE_DIR = locate_workspace()
os.chdir(WORKSPACE_DIR)
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))

# Canonical paths used across all nine notebooks.
RAW_DIR       = WORKSPACE_DIR / "dataset" / "raw"
PROCESSED_DIR = WORKSPACE_DIR / "dataset" / "processed"
QG_DIR        = PROCESSED_DIR / "question_generator"
SPLIT_DIR     = PROCESSED_DIR / "splits"
TOKENIZER_DIR = WORKSPACE_DIR / "tokenizer"
CKPT_DIR      = WORKSPACE_DIR / "checkpoints"
MODEL_DIR     = WORKSPACE_DIR / "models"
REPORTS_DIR   = WORKSPACE_DIR / "reports"
FIGURES_DIR   = REPORTS_DIR / "figures"

for d in (RAW_DIR, PROCESSED_DIR, SPLIT_DIR, TOKENIZER_DIR, CKPT_DIR,
          MODEL_DIR, REPORTS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Workspace        : {WORKSPACE_DIR}")
print(f"Python           : {sys.version.split()[0]}")
print(f"Run started      : {datetime.now(timezone.utc).isoformat(timespec='seconds')}")

---

## Step 0b — Figure and statistics conventions

One style definition serves every figure in the nine-notebook pipeline, so
charts are directly comparable when placed side by side in the write-up.

Three conventions are fixed here:

1. **A colour-blind-safe categorical palette** — the same six colours, in the
   same order, wherever a chart encodes categories.
2. **Automatic figure export** — `save_figure()` writes every figure to
   `reports/figures/` at 200 dpi with a numbered filename, and prints its
   caption, so figures can be cited as *Figure N.k* in the dissertation.
3. **A single summary-statistics function** — `describe_series()` reports
   n, mean, sd, the five-number summary, skewness and kurtosis in a fixed
   order for every variable, so distributions are described consistently.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 0b — Plotting conventions
#
# One style definition for every figure in the pipeline, so figures across the
# nine notebooks are directly comparable in the dissertation. Every figure is
# also saved to reports/figures/ at 200 dpi, ready to drop into the write-up.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.edgecolor": "#444444",
    "grid.alpha": 0.3,
    "legend.frameon": True,
    "figure.autolayout": False,
})

# Colour-blind-safe categorical palette, used consistently for every chart.
PALETTE = ["#3B6FD4", "#E1893B", "#3EA37A", "#C4576B", "#7B5EA7", "#8C7B68"]
sns.set_palette(PALETTE)

_figure_index = {"n": 0}

def save_figure(fig, slug: str, caption: str = "") -> Path:
    """Save a figure with a numbered filename and print its caption."""
    _figure_index["n"] += 1
    n = _figure_index["n"]
    path = FIGURES_DIR / f"{NOTEBOOK_ID:02d}_fig{n:02d}_{slug}.png"
    fig.savefig(path)
    label = f"Figure {NOTEBOOK_ID}.{n}"
    if caption:
        print(f"{label}: {caption}")
    print(f"           saved -> {path.relative_to(WORKSPACE_DIR)}")
    return path

def describe_series(series: pd.Series, name: str) -> pd.Series:
    """Summary statistics reported in a consistent order for every variable."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    return pd.Series({
        "n": len(s),
        "mean": s.mean(),
        "std": s.std(ddof=1),
        "min": s.min(),
        "q1": s.quantile(0.25),
        "median": s.median(),
        "q3": s.quantile(0.75),
        "max": s.max(),
        "skew": s.skew(),
        "kurtosis": s.kurtosis(),
    }, name=name)

print("Plot style       : configured")
print(f"Figure output    : {FIGURES_DIR.relative_to(WORKSPACE_DIR)}")
print(f"Palette          : {len(PALETTE)} colour-blind-safe categories")

---

## Step 1 — Load the Stage 5 results

Only candidates that were actually trained enter the comparison. A candidate
skipped for hardware reasons has **no validation loss**, and scoring it as
though it performed badly would be a false result.

In [ ]:
TRAINING_REPORT = REPORTS_DIR / "candidate_training_report.json"
assert TRAINING_REPORT.exists(), (
    f"{TRAINING_REPORT.name} missing — run Notebook 05 first."
)

report = json.loads(TRAINING_REPORT.read_text(encoding="utf-8"))
all_candidates = report["candidates"]
histories = report.get("histories", {})

trained = [c for c in all_candidates if c.get("trained")]
untrained = [c for c in all_candidates if not c.get("trained")]

print("STAGE 5 RESULTS")
print("=" * 78)
print(f"  Candidates defined : {len(all_candidates)}")
print(f"  Trained            : {len(trained)}")
print(f"  Not evaluated      : {len(untrained)}")
print("=" * 78)

if untrained:
    print("\nEXCLUDED FROM SELECTION (never measured — not poor performers):")
    for c in untrained:
        print(f"  {c['label']:24s} {c.get('skip_reason', 'no reason recorded')}")

assert trained, (
    "No candidate was trained, so no architecture can be selected. "
    "Re-run Notebook 05 on hardware that can train at least one candidate."
)

print(f"\nENTERING SELECTION ({len(trained)} candidates)")
print("-" * 78)
for c in trained:
    print(f"  {c['label']:24s} val loss {c['best_val_loss']:.4f}  "
          f"{c['parameters'] / 1e6:5.2f}M params  "
          f"gap {c['generalisation_gap']:+.4f}")

---

## Step 2 — Compute the four criteria

Three come straight from Stage 5. **Convergence stability** is computed here as
the standard deviation of the validation loss over the final epochs: a model
still oscillating at the end of training has not settled, so its best epoch may
be a fluctuation rather than a genuine optimum.

In [ ]:
def convergence_instability(history: dict, tail: int = 3) -> float:
    """Std. dev. of validation loss over the last `tail` epochs. Lower is better."""
    losses = history.get("val_loss", [])
    if len(losses) < 2:
        return 0.0
    window = losses[-min(tail, len(losses)):]
    return float(np.std(window, ddof=0))

criteria_rows = []
for candidate in trained:
    history = histories.get(candidate["candidate_id"], {})
    criteria_rows.append({
        "candidate_id": candidate["candidate_id"],
        "label": candidate["label"],
        "hypothesis": candidate.get("hypothesis", ""),
        # Raw criterion values, all "lower is better".
        "val_loss": float(candidate["best_val_loss"]),
        "gen_gap": abs(float(candidate["generalisation_gap"])),
        "parameters": int(candidate["parameters"]),
        "instability": convergence_instability(history),
        # Context, not scored.
        "val_perplexity": float(candidate["best_val_perplexity"]),
        "epochs_run": int(candidate["epochs_run"]),
        "best_epoch": int(candidate["best_epoch"]),
        "train_seconds": float(candidate["train_seconds"]),
    })

criteria = pd.DataFrame(criteria_rows)

print("RAW CRITERION VALUES (lower is better for all four)")
print("=" * 96)
print(criteria[["label", "val_loss", "gen_gap", "parameters", "instability"]]
      .to_string(index=False))
print("=" * 96)
print("\n  val_loss    : best validation cross-entropy reached")
print("  gen_gap     : |validation loss − training loss| at the best epoch")
print("  parameters  : trainable parameters (inference cost proxy)")
print("  instability : std. dev. of validation loss over the final 3 epochs")

---

## Step 3 — Normalise and score

Min–max normalisation maps each criterion to [0, 1], where **0 is best**. This
matters: without it, `parameters` (in the millions) would swamp `val_loss` (order
1) regardless of the declared weights.

Where every candidate shares the same value for a criterion, that criterion is
uninformative and is assigned 0 for all — it cannot break the tie in either
direction.

In [ ]:
WEIGHTS = {
    "val_loss": 0.45,
    "gen_gap": 0.25,
    "parameters": 0.15,
    "instability": 0.15,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9

def min_max_normalise(series: pd.Series) -> pd.Series:
    """Map to [0, 1] where 0 is best. Constant input -> all zeros."""
    low, high = series.min(), series.max()
    if not np.isfinite(low) or not np.isfinite(high) or high - low < 1e-12:
        return pd.Series(0.0, index=series.index)
    return (series - low) / (high - low)

for criterion in WEIGHTS:
    criteria[f"norm_{criterion}"] = min_max_normalise(criteria[criterion])

criteria["penalty"] = sum(
    criteria[f"norm_{criterion}"] * weight
    for criterion, weight in WEIGHTS.items()
)
# Report as a 0-100 score where higher is better, which reads more naturally.
criteria["score"] = ((1.0 - criteria["penalty"]) * 100).round(2)
criteria = criteria.sort_values("score", ascending=False).reset_index(drop=True)
criteria["rank"] = criteria.index + 1

print("NORMALISED CRITERIA (0 = best, 1 = worst within this cohort)")
print("=" * 92)
print(criteria[["rank", "label"] + [f"norm_{c}" for c in WEIGHTS] +
               ["score"]].round(4).to_string(index=False))
print("=" * 92)

print("\nWEIGHT CONTRIBUTION PER CANDIDATE (penalty points, lower is better)")
print("=" * 92)
contribution = pd.DataFrame({"label": criteria["label"]})
for criterion, weight in WEIGHTS.items():
    contribution[criterion] = (criteria[f"norm_{criterion}"] * weight).round(4)
contribution["total_penalty"] = criteria["penalty"].round(4)
print(contribution.to_string(index=False))
print("=" * 92)

In [ ]:
# ── Figure 6.1 — the scorecard ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

bars = axes[0].barh(criteria["label"][::-1], criteria["score"][::-1],
                    color=[PALETTE[2]] + [PALETTE[0]] * (len(criteria) - 1))
axes[0].set_title("Composite selection score (higher is better)")
axes[0].set_xlabel("Score / 100")
axes[0].bar_label(bars, fmt="%.1f", padding=3, fontsize=9)
axes[0].margins(x=0.16)
axes[0].annotate("winner", xy=(criteria["score"].iloc[0],
                               len(criteria) - 1),
                 xytext=(-46, 0), textcoords="offset points",
                 fontsize=8, fontweight="bold", color="white", va="center")

# Stacked penalties show *why* each candidate scored as it did.
bottom = np.zeros(len(criteria))
for i, (criterion, weight) in enumerate(WEIGHTS.items()):
    values = (criteria[f"norm_{criterion}"] * weight).values
    axes[1].bar(criteria["label"], values, bottom=bottom,
                label=f"{criterion} (w={weight})",
                color=PALETTE[i % len(PALETTE)])
    bottom += values
axes[1].set_title("Where each candidate lost points")
axes[1].set_ylabel("Weighted penalty (lower is better)")
axes[1].tick_params(axis="x", rotation=22, labelsize=8)
axes[1].legend(fontsize=7.5, loc="upper left")

# The efficiency frontier: loss against cost.
axes[2].scatter(criteria["parameters"] / 1e6, criteria["val_loss"],
                s=criteria["score"] * 3.4,
                c=[PALETTE[2] if r == 1 else PALETTE[0]
                   for r in criteria["rank"]],
                alpha=0.85, edgecolors="white", linewidths=1.8)
for _, row in criteria.iterrows():
    axes[2].annotate(f"{row['label']}\n(rank {int(row['rank'])})",
                     (row["parameters"] / 1e6, row["val_loss"]),
                     textcoords="offset points", xytext=(0, 17),
                     ha="center", fontsize=7.5)
axes[2].set_title("Efficiency frontier  (marker area ∝ score)")
axes[2].set_xlabel("Parameters (millions)")
axes[2].set_ylabel("Best validation loss")
axes[2].margins(0.3)

fig.suptitle("Multi-criteria architecture selection", y=1.03, fontsize=14,
             fontweight="bold")
fig.tight_layout()
save_figure(fig, "scorecard",
            "Centre panel is the accountability view: it shows which criterion "
            "cost each candidate its points, so the ranking is not a black box.")
plt.show()

---

## Step 4 — Radar chart: the shape of each candidate's strengths

The bar chart gives the ranking; the radar chart gives the **profile**. Axes are
oriented so *outward is better* on every one, which makes a well-rounded
candidate visibly larger in area than a specialist that wins one axis and loses
the rest.

In [ ]:
# ── Figure 6.2 — radar comparison ──────────────────────────────────────────
axis_labels = ["Validation\nquality", "Generalisation", "Parameter\nefficiency",
               "Convergence\nstability"]
angles = np.linspace(0, 2 * np.pi, len(axis_labels), endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.8),
                         subplot_kw=dict(projection="polar"))

for ax_index, ax in enumerate(axes):
    show = criteria if ax_index == 0 else criteria.head(2)
    for i, (_, row) in enumerate(show.iterrows()):
        # Invert the normalised penalties so outward = better.
        values = [1.0 - row[f"norm_{c}"] for c in WEIGHTS]
        values += values[:1]
        colour = PALETTE[2] if row["rank"] == 1 else PALETTE[i % len(PALETTE)]
        ax.plot(angles, values, linewidth=2.3, color=colour,
                label=f"{row['label']} ({row['score']:.1f})")
        ax.fill(angles, values, alpha=0.14, color=colour)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(axis_labels, fontsize=8.5)
    ax.set_ylim(0, 1.05)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=7)
    ax.set_title("All candidates" if ax_index == 0 else "Winner vs runner-up",
                 pad=26, fontsize=12)
    ax.legend(loc="upper right", bbox_to_anchor=(1.32, 1.14), fontsize=8)

fig.suptitle("Candidate profiles — outward is better on every axis", y=1.04,
             fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "radar_profiles",
            "A larger enclosed area means a better-rounded candidate. A "
            "candidate that wins one axis but collapses on another is visible "
            "immediately.")
plt.show()

---

## Step 5 — Select the winner, and check the margin

The margin over the runner-up is as important as the winner's identity. If the
top two are within a few points, the choice is close and that should be stated
plainly rather than presented as a decisive result.

In [ ]:
winner = criteria.iloc[0]
runner_up = criteria.iloc[1] if len(criteria) > 1 else None

print("=" * 80)
print("SELECTED ARCHITECTURE")
print("=" * 80)
print(f"  Candidate            : {winner['label']}")
print(f"  Identifier           : {winner['candidate_id']}")
print(f"  Hypothesis tested    : {winner['hypothesis']}")
print()
print(f"  Composite score      : {winner['score']:.2f} / 100")
print(f"  Validation loss      : {winner['val_loss']:.4f}")
print(f"  Validation perplexity: {winner['val_perplexity']:.2f}")
print(f"  Generalisation gap   : {winner['gen_gap']:.4f}")
print(f"  Parameters           : {winner['parameters'] / 1e6:.2f}M")
print(f"  Best epoch           : {int(winner['best_epoch'])} of "
      f"{int(winner['epochs_run'])} run")
print(f"  Convergence std.dev. : {winner['instability']:.5f}")
print("=" * 80)

if runner_up is not None:
    margin = winner["score"] - runner_up["score"]
    loss_delta = runner_up["val_loss"] - winner["val_loss"]
    param_ratio = runner_up["parameters"] / max(winner["parameters"], 1)

    print(f"\nMARGIN OVER RUNNER-UP ({runner_up['label']})")
    print("-" * 80)
    print(f"  Composite score margin : {margin:+.2f} points")
    print(f"  Validation loss margin : {loss_delta:+.4f}")
    print(f"  Parameter ratio        : runner-up is "
          f"{param_ratio:.2f}x the winner's size")

    if margin < 5:
        print("\n  NARROW MARGIN. The two architectures are close, so the")
        print("  selection is not strongly evidenced by these results. The")
        print("  winner is preferred on the declared weighting, but a different")
        print("  weighting could plausibly reverse the outcome. This is stated")
        print("  rather than presented as a decisive finding.")
    else:
        print(f"\n  CLEAR MARGIN ({margin:.1f} points). The ranking is robust to")
        print("  small changes in the criterion weights — see the sensitivity")
        print("  analysis in Step 6.")

    if loss_delta < 0.02 and param_ratio > 1.5:
        print("\n  Note: the runner-up is substantially larger for a negligible")
        print("  loss difference, which is exactly the trade the parameter-")
        print("  efficiency criterion exists to catch.")

---

## Step 6 — Sensitivity analysis: would a different weighting change the winner?

The declared weights are a judgement call, so their influence is tested. Four
alternative weightings — each emphasising a different criterion — are applied,
and the resulting rankings compared.

**If the same candidate wins under every weighting, the selection is robust.**
If the winner changes, that is a genuine limitation and is reported as one.

In [ ]:
ALTERNATIVE_WEIGHTS = {
    "declared": WEIGHTS,
    "loss only": {"val_loss": 1.0, "gen_gap": 0.0, "parameters": 0.0,
                  "instability": 0.0},
    "generalisation first": {"val_loss": 0.30, "gen_gap": 0.50,
                             "parameters": 0.10, "instability": 0.10},
    "efficiency first": {"val_loss": 0.30, "gen_gap": 0.15,
                         "parameters": 0.45, "instability": 0.10},
    "equal weights": {c: 0.25 for c in WEIGHTS},
}

sensitivity = pd.DataFrame({"label": criteria["label"]})
winners = {}
for scheme, weights in ALTERNATIVE_WEIGHTS.items():
    penalty = sum(criteria[f"norm_{c}"] * w for c, w in weights.items())
    score = (1.0 - penalty) * 100
    sensitivity[scheme] = score.round(2)
    winners[scheme] = criteria.loc[score.idxmax(), "label"]

print("SENSITIVITY ANALYSIS — score under each weighting scheme")
print("=" * 92)
print(sensitivity.to_string(index=False))
print("=" * 92)
print("\nWINNER UNDER EACH SCHEME")
print("-" * 60)
for scheme, winning_label in winners.items():
    marker = "" if winning_label == winner["label"] else "   <- DIFFERENT"
    print(f"  {scheme:22s} {winning_label}{marker}")

distinct_winners = set(winners.values())
print("-" * 60)
if len(distinct_winners) == 1:
    print(f"\nROBUST: {winner['label']} wins under all "
          f"{len(ALTERNATIVE_WEIGHTS)} weighting schemes.")
    print("The selection does not depend on the particular weights chosen.")
else:
    print(f"\nSENSITIVE: {len(distinct_winners)} different candidates win "
          f"depending on the weighting.")
    print(f"Winners: {sorted(distinct_winners)}")
    print(f"\n{winner['label']} is carried forward on the declared weighting,")
    print("which was fixed before the scores were computed. This sensitivity is")
    print("a real limitation of the selection and is reported as such.")

In [ ]:
# ── Figure 6.3 — sensitivity heatmap ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15.5, 4.8),
                         gridspec_kw={"width_ratios": [1.25, 1]})

heat = sensitivity.set_index("label")
sns.heatmap(heat, annot=True, fmt=".1f", cmap="RdYlGn", center=heat.values.mean(),
            linewidths=0.7, linecolor="white", ax=axes[0],
            cbar_kws={"label": "Score / 100"}, annot_kws={"size": 8.5})
axes[0].set_title("Score under each weighting scheme")
axes[0].set_xlabel("Weighting scheme")
axes[0].set_ylabel("")
plt.setp(axes[0].get_xticklabels(), rotation=26, ha="right", fontsize=8)

# Rank stability is the clearer view: flat lines mean a robust ranking.
ranks = heat.rank(ascending=False, axis=0)
for i, label in enumerate(ranks.index):
    axes[1].plot(range(len(ranks.columns)), ranks.loc[label],
                 marker="o", markersize=7, linewidth=2.1,
                 color=PALETTE[i % len(PALETTE)], label=label)
axes[1].set_xticks(range(len(ranks.columns)))
axes[1].set_xticklabels(ranks.columns, rotation=26, ha="right", fontsize=8)
axes[1].set_yticks(range(1, len(ranks) + 1))
axes[1].invert_yaxis()
axes[1].set_ylabel("Rank (1 = best)")
axes[1].set_title("Rank stability across weightings")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

fig.suptitle("Is the selection robust to the choice of weights?", y=1.04,
             fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "sensitivity",
            "Right panel: a flat line means that candidate holds its rank "
            "regardless of weighting, which is what makes the selection "
            "defensible.")
plt.show()

---

## Step 7 — Record the selection

In [ ]:
selection = {
    "stage": "06_model_comparison_and_selection",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "data_used": "validation split only — the test split was not read",
    "selected": {
        "candidate_id": winner["candidate_id"],
        "label": winner["label"],
        "hypothesis": winner["hypothesis"],
        "composite_score": float(winner["score"]),
        "val_loss": float(winner["val_loss"]),
        "val_perplexity": float(winner["val_perplexity"]),
        "generalisation_gap": float(winner["gen_gap"]),
        "parameters": int(winner["parameters"]),
        "best_epoch": int(winner["best_epoch"]),
        "checkpoint": str((CKPT_DIR / winner["candidate_id"])
                          .relative_to(WORKSPACE_DIR)),
    },
    "criteria_weights": WEIGHTS,
    "scorecard": criteria.drop(columns=["hypothesis"]).round(5)
                         .to_dict(orient="records"),
    "margin_over_runner_up": (
        float(winner["score"] - runner_up["score"]) if runner_up is not None
        else None),
    "sensitivity": {
        "schemes": {k: {c: float(v) for c, v in w.items()}
                    for k, w in ALTERNATIVE_WEIGHTS.items()},
        "winner_per_scheme": winners,
        "robust": len(distinct_winners) == 1,
        "distinct_winners": sorted(distinct_winners),
    },
    "excluded_candidates": [
        {"label": c["label"], "reason": c.get("skip_reason", "not trained")}
        for c in untrained
    ],
}

selection_path = REPORTS_DIR / "model_selection.json"
selection_path.write_text(json.dumps(selection, indent=2, default=str),
                          encoding="utf-8")

print(f"Selection record : {selection_path.relative_to(WORKSPACE_DIR)}")
print(f"Winner           : {winner['label']} ({winner['candidate_id']})")
print(f"Robust selection : {len(distinct_winners) == 1}")
print(f"Figures          : "
      f"{len(sorted(FIGURES_DIR.glob(f'{NOTEBOOK_ID:02d}_*.png')))}")

---

## Stage 6 summary

| Aspect | Approach |
|---|---|
| Data used | validation split only — no test data touched |
| Criteria | four, min–max normalised, weights declared in advance |
| Accountability | Figure 6.1 shows which criterion cost each candidate points |
| Profile view | Figure 6.2 radar — outward is better on every axis |
| Robustness | Figure 6.3 — five weighting schemes, rank stability tested |
| Excluded candidates | recorded with reasons, never scored as failures |

### Why this is defensible

Three properties, in order of importance:

1. **The weights were fixed before the scores were computed**, so they could not
   be tuned to produce a preferred winner.
2. **The margin is reported honestly.** Step 5 states plainly when the top two
   are close, rather than presenting a narrow win as decisive.
3. **The sensitivity analysis is reported whichever way it comes out.** If the
   winner changes under a different weighting, that is printed as a limitation.

### Next

**Notebook 07 — Specialisation of the Selected Model**, which continues training
the winner on the task-specific objective.